# Grinding path planning from sparse PointCNN output

`Weld_Recognition&Grinding_Path_Planning(PointCNN).ipynb` extracts the path by rasterizing the bead points into a 150x150 (1px=1mm) image and eroding/dilating/skeletonizing it -- that needs a solid, densely-populated blob. `PointCNN_Inference.ipynb`'s output is sparse even with many voted FPS passes (the checkpoint was trained on single 2048-point global downsamples of the whole scan, not dense local patches), so the raster step produces an empty skeleton on real inference output.

This notebook reuses the same building blocks as the original (voxel filtering, B-spline surface fit + weld bead height, greedy nearest-neighbor path ordering, Delaunay surface normals, normal smoothing, tangent/normal -> quaternion) but applies the path-ordering + B-spline step directly to the labeled 3D points instead of to raster-derived pixel coordinates. Everything through the surface fit and height check is copy-pasted from `Weld_Recognition&Grinding_Path_Planning(PointCNN).ipynb` cells 4-5 unchanged; only the path-extraction step (normally cells 6-14, the raster/skeleton path) is new here.

Output is in the **scanner's coordinate frame**, not the robot's -- cell 23's `transformation_matrix` in the original notebook is specific to this project's ABB eye-to-hand calibration and is not applied here.

In [ ]:
import time
import numpy as np
import pandas as pd
import os
os.environ.pop('WAYLAND_DISPLAY', None)  # Open3D's GLFW window fails to open under native Wayland on this machine (GLEW init error) -- force XWayland instead
os.environ['XDG_SESSION_TYPE'] = 'x11'
import open3d as o3d
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.cluster import DBSCAN
from scipy.interpolate import SmoothBivariateSpline
from scipy.spatial import KDTree, Delaunay, distance_matrix
from scipy.ndimage import gaussian_filter1d
from scipy.spatial.transform import Rotation as R
import scipy.interpolate as si

# Load segmented point cloud

In [ ]:
point_file = 'outputs/predicted_3.txt'  # output of PointCNN_Inference.ipynb
cloud = np.loadtxt(point_file)
point_cloud = np.array(cloud[:, :3]).reshape(-1, 3)
truth_label = np.array(cloud[:, 3])

points_label0 = point_cloud[truth_label == 0]
points_label1 = point_cloud[truth_label == 1]

# Same DBSCAN cleanup as the original notebook's cell 2, on the work-piece side only.
# eps=2.0, min_samples=1000 (the original default) marks real scans as all-noise --
# min_samples needs to be well below the local point density. Check yours:
tree = KDTree(points_label0)
local_density = np.median(tree.query_ball_point(points_label0[::50], r=2.0, return_length=True))
print(f"Median neighbor count within r=2mm: {local_density:.0f} (min_samples below should be well under this)")

dbscan = DBSCAN(eps=2.0, min_samples=max(10, int(local_density * 0.3)))
labels_dbscan = dbscan.fit_predict(points_label0)
points_label0 = points_label0[labels_dbscan != -1]

print(f"work piece (label 0): {len(points_label0)} points")
print(f"weld bead (label 1): {len(points_label1)} points")

In [ ]:
fig = plt.figure(figsize=(16, 7))
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(points_label0[:, 0], points_label0[:, 1], points_label0[:, 2], c='lightblue', s=1, alpha=0.15, label='work piece')
ax1.scatter(points_label1[:, 0], points_label1[:, 1], points_label1[:, 2], c='red', s=15, label='bead (raw)')
ax1.set_xlabel('X(mm)'); ax1.set_ylabel('Y(mm)'); ax1.set_zlabel('Z(mm)')
ax1.set_title('3D view')
ax1.legend()

ax2 = fig.add_subplot(122)
ax2.scatter(points_label0[:, 0], points_label0[:, 1], c='lightblue', s=1, alpha=0.15, label='work piece')
ax2.scatter(points_label1[:, 0], points_label1[:, 1], c='red', s=15, label='bead (raw)')
ax2.set_xlabel('X(mm)'); ax2.set_ylabel('Y(mm)')
ax2.set_title('Top-down (X-Y) view')
ax2.axis('equal')
ax2.legend()
plt.tight_layout()
plt.show()

# Voxel filter the work-piece surface

Unchanged from `Weld_Recognition&Grinding_Path_Planning(PointCNN).ipynb` cell 4.

In [ ]:
time3 = time.time()

weld_bead = points_label1
work_piece = points_label0

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.set_xlabel("X(mm)")
ax.set_ylabel("Y(mm)")
ax.set_zlabel("Z(mm)")
ax.scatter(work_piece[:, 0], work_piece[:, 1], work_piece[:, 2], c='blue', s=10)
plt.show()

filtered_points = []
tempx, tempy, tempz = [], [], []
for i in range(0, len(work_piece)):
    tempx.append(work_piece[i][0])
    tempy.append(work_piece[i][1])
    tempz.append(work_piece[i][2])

x_max, y_max, z_max = np.max(tempx), np.max(tempy), np.max(tempz)
x_min, y_min, z_min = np.min(tempx), np.min(tempy), np.min(tempz)

size_v = 2
print("Voxel size =", str(size_v))

d_x = (x_max - x_min) / size_v
d_y = (y_max - y_min) / size_v
d_z = (z_max - z_min) / size_v

h = []
for i in range(0, len(work_piece)):
    hx = np.floor((work_piece[i][0] - x_min) / size_v)
    hy = np.floor((work_piece[i][1] - y_min) / size_v)
    hz = np.floor((work_piece[i][2] - z_min) / size_v)
    h.append(hx + hy * d_x + hz * d_x * d_y)

h = np.array(h)
index_h = np.argsort(h)
h_sorted = h[index_h]
count = 0
np.seterr(divide='ignore', invalid='ignore')

for i in range(0, len(h_sorted) - 1):
    if h_sorted[i] == h_sorted[i + 1]:
        continue
    index_point = index_h[count:i + 1]
    x_temp, y_temp, z_temp = [], [], []
    for p in index_point:
        x_temp.append(work_piece[p][0])
        y_temp.append(work_piece[p][1])
        z_temp.append(work_piece[p][2])
    filtered_points.append([np.mean(x_temp), np.mean(y_temp), np.mean(z_temp)])
    count = i

filtered_points = np.array(filtered_points, dtype=np.float64)

x_f, y_f, z_f = [], [], []
for i in range(0, len(filtered_points)):
    x_f.append(filtered_points[i][0])
    y_f.append(filtered_points[i][1])
    z_f.append(filtered_points[i][2])

while (0 in x_f):
    x_f.remove(0)
    y_f.remove(0)
    z_f.remove(0)

print(str(len(x_f)), "points in total.")

work_piece = np.array(list(zip(x_f, y_f, z_f)))

point_cloud_o3d = o3d.geometry.PointCloud()
point_cloud_o3d.points = o3d.utility.Vector3dVector(work_piece)
cl, ind = point_cloud_o3d.remove_statistical_outlier(nb_neighbors=5, std_ratio=1.5)
work_piece = np.asarray(cl.points)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.set_xlabel("X(mm)")
ax.set_ylabel("Y(mm)")
ax.set_zlabel("Z(mm)")
ax.scatter(work_piece[:, 0], work_piece[:, 1], work_piece[:, 2], c='blue', s=10)
plt.show()

time4 = time.time()
t2 = np.round((time4 - time3), 4)
print("Voxel filtering time: ", t2, "s")

# Surface fit + weld bead height

Unchanged from `Weld_Recognition&Grinding_Path_Planning(PointCNN).ipynb` cell 5. Produces `spline`/`X`/`Y`/`Z` (used for surface normals below) and prints the bead height.

In [ ]:
time5 = time.time()

x = work_piece[:, 0]
y = work_piece[:, 1]
z = work_piece[:, 2]

x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()

spline = SmoothBivariateSpline(x, y, z, s=50)

X, Y = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = spline.ev(X.ravel(), Y.ravel()).reshape(300, 300)

dZ_dx = spline.ev(X.ravel(), Y.ravel(), dx=1, dy=0).reshape(300, 300)
dZ_dy = spline.ev(X.ravel(), Y.ravel(), dx=0, dy=1).reshape(300, 300)
surface_normals_grid = np.dstack((-dZ_dx, -dZ_dy, np.ones_like(Z)))
surface_normals_grid /= np.linalg.norm(surface_normals_grid, axis=2, keepdims=True)

weld_bead_tree = KDTree(weld_bead)
offset_distance = 0.1
max_offset = 100
current_offset = 1
last_offset_point = []

while current_offset < max_offset:
    Z_offset = Z + current_offset * surface_normals_grid[..., 2]
    X_offset = X + current_offset * surface_normals_grid[..., 0]
    Y_offset = Y + current_offset * surface_normals_grid[..., 1]

    surface_points = np.column_stack((X_offset.ravel(), Y_offset.ravel(), Z_offset.ravel()))
    dist, _ = weld_bead_tree.query(surface_points)

    if np.all(dist > offset_distance):
        break
    last_offset_point = surface_points[dist <= offset_distance]
    current_offset += offset_distance

original_points = np.column_stack((X.ravel(), Y.ravel(), Z.ravel()))
original_tree = KDTree(original_points)

if len(last_offset_point) > 0:
    last_offset_point = np.array(last_offset_point)
    weld_bead_distances, nearest_indices = weld_bead_tree.query(last_offset_point)
    closest_weld_bead_points = weld_bead[nearest_indices]
    closest_original_distances, closest_original_indices = original_tree.query(closest_weld_bead_points)
    min_distance = np.round(closest_original_distances.min(), 3)
    print("Weld bead height:", min_distance, "mm")
else:
    # Known bug carried over from the original notebook: if the offset search never gets
    # within offset_distance of the bead, last_offset_point (and therefore min_distance)
    # is never set. Fixed here instead of silently crashing downstream.
    print("Weld bead height: could not be determined (offset search found no collision)")
    min_distance = None

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x, y, z, c='blue', s=5)
ax.plot_surface(X, Y, Z, color='r', alpha=0.5)
ax.set_xlabel('X(mm)')
ax.set_ylabel('Y(mm)')
ax.set_zlabel('Z(mm)')
plt.title('B-Spline Surface Fitting with Minimum Distance to Weld Bead')
plt.show()

time6 = time.time()
t3 = np.round((time6 - time5), 4)
print("Weld bead height computation time: ", t3, "s")

# Filter sparse bead points and order into a path

**New in this notebook** -- not in the original. Drops isolated bead points (the sparse false positives a low-pass-count inference run leaves behind), then reuses the exact greedy nearest-neighbor ordering algorithm from cell 12 of the original notebook, applied to the 3D bead points directly instead of to raster/skeleton pixel coordinates.

In [ ]:
outlier_radius = 15.0     # mm
outlier_min_neighbors = 3

# Batches without cross-batch voting can independently pick the same point
# via FPS more than once (observed on real inference output: 152 raw bead
# points, only 123 unique) -- exact duplicates break splprep's parametrization
# below, so drop them here regardless of the outlier filter.
weld_bead_unique = np.unique(weld_bead, axis=0)
print(f"bead points: {len(weld_bead)} -> {len(weld_bead_unique)} after de-duplication")

tree = KDTree(weld_bead_unique)
neighbor_counts = tree.query_ball_point(weld_bead_unique, r=outlier_radius, return_length=True)
bead_points = weld_bead_unique[neighbor_counts >= outlier_min_neighbors]
print(f"bead points: {len(weld_bead_unique)} -> {len(bead_points)} after outlier filter")

# --- same greedy nearest-neighbor ordering as cell 12, run on 3D points instead of path_2D ---
def calculate_total_distance(points, order):
    distances = distance_matrix(points[order], points[order])
    return sum(distances[i, i + 1] for i in range(len(order) - 1))

def reorder_weld_bead(points, start_index):
    num_points = len(points)
    distances = distance_matrix(points, points)
    reordered_indices = [start_index]
    remaining_indices = set(range(num_points))
    remaining_indices.remove(start_index)
    while remaining_indices:
        last_index = reordered_indices[-1]
        nearest_index = min(remaining_indices, key=lambda x: distances[last_index, x])
        reordered_indices.append(nearest_index)
        remaining_indices.remove(nearest_index)
    return reordered_indices

min_total_distance = float('inf')
best_order = None
for i in range(len(bead_points)):
    current_order = reorder_weld_bead(bead_points, i)
    current_distance = calculate_total_distance(bead_points, current_order)
    if current_distance < min_total_distance:
        min_total_distance = current_distance
        best_order = current_order

reordered_points = bead_points[best_order]
print("Greedy path length:", round(min_total_distance, 2), "mm")

fig = plt.figure(figsize=(14, 6))
ax = fig.add_subplot(121, projection='3d')
ax.scatter(reordered_points[0, 0], reordered_points[0, 1], reordered_points[0, 2], c='red', label='First point')
ax.scatter(reordered_points[1:-1, 0], reordered_points[1:-1, 1], reordered_points[1:-1, 2], c='orange', label='Middle points')
ax.scatter(reordered_points[-1, 0], reordered_points[-1, 1], reordered_points[-1, 2], c='green', label='Last point')
ax.plot(reordered_points[:, 0], reordered_points[:, 1], reordered_points[:, 2], c='gray', alpha=0.5)
ax.set_xlabel('X(mm)'); ax.set_ylabel('Y(mm)'); ax.set_zlabel('Z(mm)')
ax.set_title('3D view')
ax.legend()

ax2 = fig.add_subplot(122)
ax2.plot(reordered_points[:, 0], reordered_points[:, 1], c='gray', alpha=0.5)
ax2.scatter(reordered_points[0, 0], reordered_points[0, 1], c='red', label='First point', zorder=3)
ax2.scatter(reordered_points[1:-1, 0], reordered_points[1:-1, 1], c='orange', label='Middle points', zorder=3)
ax2.scatter(reordered_points[-1, 0], reordered_points[-1, 1], c='green', label='Last point', zorder=3)
ax2.set_xlabel('X(mm)'); ax2.set_ylabel('Y(mm)')
ax2.set_title('Top-down (X-Y) view -- check the ordering follows the true bead shape (e.g. a U)')
ax2.axis('equal')
ax2.legend()
plt.tight_layout()
plt.title('Ordered bead points')
plt.show()

# B-spline path fit

Same `splprep`/`splev` call as cell 14 (`s=2, k=3`), fit directly on the 3D points -- no Douglas-Peucker simplification since there are already few points, no 2D->3D projection since these are already real 3D points from the scan.

In [ ]:
num_samples = 100
degree = min(3, len(reordered_points) - 1)
tck, u = si.splprep(reordered_points.T, s=2, k=degree)
uu = np.linspace(0, 1, num_samples)
path_3D = np.array(si.splev(uu, tck)).T

arc_length = np.sum(np.linalg.norm(np.diff(path_3D, axis=0), axis=1))
print(f"path_3D: {path_3D.shape}, arc length: {arc_length:.2f} mm")

fig = plt.figure(figsize=(14, 6))
ax = fig.add_subplot(121, projection='3d')
ax.plot(path_3D[:, 0], path_3D[:, 1], path_3D[:, 2], c='black', label='B-spline path')
ax.scatter(reordered_points[:, 0], reordered_points[:, 1], reordered_points[:, 2], c='red', alpha=0.5, label='Bead points')
ax.set_xlabel('X(mm)'); ax.set_ylabel('Y(mm)'); ax.set_zlabel('Z(mm)')
ax.set_title('3D view')
ax.legend()

ax2 = fig.add_subplot(122)
ax2.plot(path_3D[:, 0], path_3D[:, 1], c='black', label='B-spline path')
ax2.scatter(reordered_points[:, 0], reordered_points[:, 1], c='red', alpha=0.5, label='Bead points')
ax2.set_xlabel('X(mm)'); ax2.set_ylabel('Y(mm)')
ax2.set_title('Top-down (X-Y) view')
ax2.axis('equal')
ax2.legend()
plt.tight_layout()
plt.show()

# Surface normals along the path

Same `compute_normal` as cell 22, using the Delaunay triangulation of the fitted surface (`spline`/`X`/`Y`/`Z` from the surface-fit cell above).

In [ ]:
points_3D_surf = np.column_stack((X.ravel(), Y.ravel(), Z.ravel()))
tri = Delaunay(points_3D_surf[:, :2])

def compute_normal(tri, points):
    normals = []
    for point in points:
        simplex = tri.find_simplex(point[:2])
        if simplex == -1:
            normals.append([0, 0, 1])
            continue
        vertices = tri.simplices[simplex]
        p0, p1, p2 = points_3D_surf[vertices]
        v1 = p1 - p0
        v2 = p2 - p0
        normal = np.cross(v1, v2)
        normal = normal / np.linalg.norm(normal)
        normals.append(normal)
    return np.array(normals)

normals = compute_normal(tri, path_3D)
print("normals:", normals.shape)

# Smooth normals + orientation (quaternions)

`smooth_normals` (3x Gaussian pass) is copied unchanged from cell 25 -- the original applies it before anything geometry-critical because per-triangle Delaunay normals can flip sign between adjacent points. `calculate_quaternions_with_non_orthogonal_normals` is copied unchanged from cell 32.

In [ ]:
def smooth_normals(normals, sigma=5):
    smoothed_normals = np.zeros_like(normals)
    for i in range(3):
        smoothed_normals[:, i] = gaussian_filter1d(normals[:, i], sigma=sigma, mode='reflect')
    norm = np.linalg.norm(smoothed_normals, axis=1, keepdims=True)
    norm[norm == 0] = 1
    smoothed_normals /= norm
    return smoothed_normals

normals = smooth_normals(normals, sigma=0.75)
normals = smooth_normals(normals, sigma=0.75)
normals = smooth_normals(normals, sigma=0.75)

def calculate_quaternions_with_non_orthogonal_normals(path_3D, normals, t_vectors):
    quaternions = []
    for t_vector, normal in zip(t_vectors, normals):
        t_vector = t_vector / np.linalg.norm(t_vector)
        normal = normal - np.dot(normal, t_vector) * t_vector
        normal = normal / np.linalg.norm(normal)
        y_axis = np.cross(normal, t_vector)
        y_axis = y_axis / np.linalg.norm(y_axis)
        rotation_matrix = np.vstack((t_vector, y_axis, normal)).T
        quat = R.from_matrix(rotation_matrix).as_quat()
        quat_reordered = [quat[3], quat[0], quat[1], quat[2]]
        quaternions.append(quat_reordered)
    return np.array(quaternions)

t_vectors = np.diff(path_3D, axis=0)
t_vectors = np.vstack([t_vectors, t_vectors[-1]])  # repeat the last tangent for the final point

quaternion = calculate_quaternions_with_non_orthogonal_normals(path_3D, normals, t_vectors)
print("quaternions:", quaternion.shape)

# Visualize trajectory

Same Open3D interactive view style as cell 25 (path + surface + normal lines), requires a real display.

In [ ]:
pcd_work_piece = o3d.geometry.PointCloud()
pcd_work_piece.points = o3d.utility.Vector3dVector(work_piece)
pcd_work_piece.paint_uniform_color([0, 0, 1])

pcd_path = o3d.geometry.PointCloud()
pcd_path.points = o3d.utility.Vector3dVector(path_3D)
pcd_path.paint_uniform_color([1, 0, 0])

lines, colors, line_points = [], [], []
for i in range(len(path_3D)):
    start_point = path_3D[i]
    end_point = start_point + normals[i] * 5  # 5mm normal length for visualization
    line_points.append(start_point)
    line_points.append(end_point)
    lines.append([2 * i, 2 * i + 1])
    colors.append([0, 0, 0])

line_set = o3d.geometry.LineSet()
line_set.points = o3d.utility.Vector3dVector(line_points)
line_set.lines = o3d.utility.Vector2iVector(lines)
line_set.colors = o3d.utility.Vector3dVector(colors)

axis = o3d.geometry.TriangleMesh.create_coordinate_frame(size=20.0, origin=[0, 0, 0])

vis = o3d.visualization.Visualizer()
vis.create_window()
vis.add_geometry(axis)
vis.add_geometry(pcd_work_piece)
vis.add_geometry(pcd_path)
vis.add_geometry(line_set)
vis.run()
vis.destroy_window()

# Save trajectory

Same column layout as `df_0` in cell 32 of the original notebook (`x, y, z, i, j, k, q1, q2, q3, q4`), still in scanner coordinates.

In [ ]:
df_path = pd.DataFrame(path_3D, columns=['x', 'y', 'z'])
df_normals = pd.DataFrame(normals, columns=['i', 'j', 'k'])
df_quaternion = pd.DataFrame(quaternion, columns=['q1', 'q2', 'q3', 'q4'])
df_0 = pd.concat([df_path, df_normals, df_quaternion], axis=1).round(4)

import os
os.makedirs('outputs', exist_ok=True)
out_path = 'outputs/trajectory.csv'
df_0.to_csv(out_path, index=False)
print(f"Wrote {out_path}: {len(df_0)} pose points")
df_0.head()